# VT-GraF Benchmark: Visible--Thermal Granular Faults under severe clutter
## Comparison: Unsupervised Generalized Frangi Graph vs. Standard Frangi + SAM 2

This notebook reproduces the comparison on the **Raphael-Dataset** (multimodal asphalt crack dataset with severe granular clutter) between:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised multimodal graph-based method (using MST + Betweenness Centrality on GPU).
2. **Baseline (Standard Frangi on Thermal + SAM 2 on Visible)**: Zero-shot SAM 2 prompted automatically by standard Frangi filter responses from the thermal modality.

---
### Technical Setup:
- Automatically downloads the **Raphael-Dataset** from Google Drive.
- Installs **Segment Anything 2 (SAM 2)** from Meta's repository and downloads the `sam2_hiera_large` weights.
- Runs both methods and outputs comparative metrics: Jaccard (IoU), Tversky index, and Wasserstein distance.


## 1. Environment Setup & Datasets
We download the dataset, install SAM 2, and retrieve the pre-trained weights.


In [ ]:
# Install required packages
!pip install -q gdown POT scikit-image opencv-python matplotlib scipy torch torchvision

# Install SAM 2 from Meta's source
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Download SAM 2 Large weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt



In [ ]:
# Download the Raphael dataset using gdown
import os
import gdown
from pathlib import Path

folder_id = '1d79CVf9Vqgwwjqn6b2gbc40eu2MM7B7-'
dest_dir = 'Raphael-Dataset'

def check_dataset_exists():
    for path in Path('.').rglob('Fissure 1'):
        return True
    return False

if not check_dataset_exists():
    print("Downloading Raphael Dataset from Google Drive...")
    gdown.download_folder(id=folder_id, output=dest_dir, quiet=False, use_cookies=False)
    print("Download completed.")
else:
    print("Raphael Dataset already present.")



## 2. Load the Generalized Frangi Graph Modules
We resolve the repository paths dynamically and load our PyTorch GPU-based implementation.


In [ ]:
import sys
import os
from pathlib import Path

# Dynamically resolve repository paths in Colab, Local, or Google Drive
possible_paths = [
    Path('.'),
    Path('../..'),
    Path('Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset'),
    Path('/content/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset'),
]

repo_root = None
for p in possible_paths:
    if (p / 'ISPRS').exists():
        repo_root = p.resolve()
        if str(repo_root) not in sys.path:
            sys.path.append(str(repo_root))
        break

if repo_root is not None:
    print(f"Added repository root to sys.path: {repo_root}")
else:
    print("WARNING: ISPRS directory not found in typical paths. Please ensure the repository is cloned properly.")

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Import our custom modules
from ISPRS.src import RaphaelDataset, extract_frangi_graph_gpu, skeletonize_lee, thicken, compute_metrics, wasserstein_distance_skeletons

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Active device:", device)

dataset = RaphaelDataset('.')



## 3. Initialize SAM 2 Predictor
We load the pre-trained SAM 2 Large model on GPU (or CPU as fallback) and initialize the image predictor.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, sam2_checkpoint, device=device))
print("SAM 2 Large predictor loaded successfully!")



## 4. Baseline Pipeline: Standard Frangi (Thermal) + SAM 2 (Visible)
This function implements the baseline method:
1. Applies a standard Frangi filter from `skimage` to the grayscale decoded thermal image.
2. Thresholds the response (taking the top 0.5% pixels) and skeletonizes it.
3. Spatially samples 12 points along the centerline as prompts.
4. Feeds the visible image and these points to SAM 2.


In [ ]:
from skimage.filters import frangi

def run_baseline_frangi_sam2(sample, predictor, num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    frangi_response = frangi(img_ir_np / 255.0, sigmas=[1, 3, 5, 7, 9])
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return pred_mask, pts



## 5. Quantitative Evaluation & Benchmark
We evaluate both methods on all 5 images of the Raphael-Dataset.
- For **Ours (Generalized Frangi Graph)**, we use parameters: `K=2` (dual triangle graph), scale set `Σ=[20,30,40]` and weights `visible: 1/3, infrared: 2/3` (the latest parameters from your notebook). Skeletons are thickened to **5 pixels** (`pixels=5`) for final evaluation.
- For the **Baseline (Frangi Thermal + SAM 2)**, we generate the mask, skeletonize it, and thicken it to **5 pixels** similarly.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[20,30,40]
    _, _, centrality_i, _, _ = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=[20, 30, 40], R=3, K=2, device=device
    )
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # --- 2. RUN BASELINE (Frangi Thermal + SAM 2) ---
    pred_baseline, sampled_pts = run_baseline_frangi_sam2(sample, predictor, num_prompts=12)
    sk_pred_baseline = skeletonize_lee(pred_baseline)
    sk_pred_thick_baseline = thicken(sk_pred_baseline, pixels=5)
    
    # --- 3. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline
    jac_base, tv_base = compute_metrics(sk_pred_thick_baseline, sk_gt_thick)
    wass_base = wasserstein_distance_skeletons(sk_pred_thick_baseline, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'Baseline_Jaccard': jac_base,
        'Baseline_Tversky': tv_base,
        'Baseline_Wasserstein': wass_base
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours': sk_pred_thick_ours,
        'baseline': sk_pred_thick_baseline,
        'points': sampled_pts
    }

df_res = pd.DataFrame(results)



## 6. Comparison Results Table


In [ ]:
# Format and display the results
pd.set_option('display.precision', 4)
display(df_res)

# Print Global Averages
print("\n" + "="*50)
print("--- SUMMARY STATISTICS ---")
print("="*50)
print(f"Jaccard (IoU) Mean   : Ours = {df_res['Ours_Jaccard'].mean():.4f} | Baseline = {df_res['Baseline_Jaccard'].mean():.4f}")
print(f"Tversky Index Mean   : Ours = {df_res['Ours_Tversky'].mean():.4f} | Baseline = {df_res['Baseline_Tversky'].mean():.4f}")
print(f"Wasserstein Dist Mean: Ours = {df_res['Ours_Wasserstein'].mean():.4f} | Baseline = {df_res['Baseline_Wasserstein'].mean():.4f}")



## 7. Visual Inspection
We plot the results side-by-side to visually analyze the structural output.


In [ ]:
for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    fig.suptitle(f"Visual Comparison - {name}", fontsize=20, fontweight='bold')
    
    # 1. Visible Image
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 0].set_title("1. Visible (Asphalt Clutter)", fontsize=14)
    axes[0, 0].axis('off')
    
    # 2. Thermal Grayscale (Decoded)
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 1].set_title("2. Thermal (Decoded Grayscale)", fontsize=14)
    axes[0, 1].axis('off')
    
    # 3. Ground Truth
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    axes[0, 2].set_title("3. Ground Truth Mask", fontsize=14)
    axes[0, 2].axis('off')
    
    # 4. Ours (Generalized Frangi Graph)
    axes[1, 0].imshow(vis_data['ours'], cmap='hot')
    axes[1, 0].set_title("4. Ours (Topological Graph Skeleton)", fontsize=14)
    axes[1, 0].axis('off')
    
    # 5. Baseline (Frangi Thermal + SAM 2)
    # Draw visible background and overlay baseline with sampled prompt points
    h, w = vis_data['visible'].shape
    rgba_vis = np.stack([vis_data['visible']]*3 + [np.ones((h, w))], axis=-1)
    rgba_base = np.zeros((h, w, 4))
    rgba_base[vis_data['baseline'] > 0] = [0.0, 1.0, 0.0, 0.5] # Green
    
    axes[1, 1].imshow(vis_data['visible'], cmap='gray')
    axes[1, 1].imshow(rgba_base)
    pts = vis_data['points']
    axes[1, 1].scatter(pts[:, 0], pts[:, 1], color='red', s=40, label='Prompts')
    axes[1, 1].set_title("5. Baseline SAM 2 (Green) with Prompts (Red)", fontsize=14)
    axes[1, 1].legend()
    axes[1, 1].axis('off')
    
    # 6. Combined Comparison Overlay
    # White = Ground Truth, Green = Ours, Red = Baseline
    rgba_gt = np.zeros((h, w, 4))
    rgba_gt[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.3]
    
    rgba_ours = np.zeros((h, w, 4))
    rgba_ours[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green
    
    rgba_b = np.zeros((h, w, 4))
    rgba_b[vis_data['baseline'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red
    
    axes[1, 2].imshow(rgba_gt)
    axes[1, 2].imshow(rgba_ours)
    axes[1, 2].imshow(rgba_b)
    axes[1, 2].set_title("6. Overlay (White: GT, Green: Ours, Red: Baseline)", fontsize=14)
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

